# Day 2 Exercise — Website Summarizer with Ollama

**Personal exercise** (fork only, not for upstream PR)

Upgrade the Day 1 website summarizer to use **Ollama locally** via the OpenAI-compatible endpoint from `day2.ipynb`.

Uses our Playwright scraper for JavaScript-heavy sites (e.g. kabbalahmedia.info).

In [ ]:
import sys

import requests
from IPython.display import Markdown, display
from openai import OpenAI

sys.path.insert(0, "../community-contributions/slavapa")
from scraper_playwright import fetch_website_contents

OLLAMA_BASE_URL = "http://localhost:11434/v1"
MODEL = "llama3.2"  # use "llama3.2:1b" on slower machines

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

In [ ]:
# Confirm Ollama is running
requests.get("http://localhost:11434").content

## Prompts (same pattern as Day 1)

In [ ]:
system_prompt = """
You are an assistant that analyzes website contents and provides a short summary.
Ignore navigation menus, footers, and cookie banners.
Respond in markdown. Do not wrap the markdown in a code block.
"""

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, summarize these too.

"""

EXERCISE_SITES = [
    "https://kabbalahmedia.info/en/",
    "https://www.michaellaitman.com/",
    "https://kabuconnect.com/",
]

In [ ]:
def messages_for(website: str) -> list[dict]:
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website},
    ]


def summarize(url: str) -> str:
    website = fetch_website_contents(url)
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=messages_for(website),
    )
    return response.choices[0].message.content


def display_summary(url: str) -> None:
    print(f"### {url}\n")
    display(Markdown(summarize(url)))

## Summarize our exercise sites with Ollama (free, local, private)

In [ ]:
for site in EXERCISE_SITES:
    display_summary(site)
    print("\n---\n")

## Try any URL

In [ ]:
display_summary("https://edwarddonner.com")